# Step 3: 用 Ollama qwen2.5:7b 批量补充中文翻译

遍历所有 JSON 文件，找到 `翻译` 为空的条目，把 `例句` 发给本地 Ollama 翻译成中文，写回原文件。

**前提**：Ollama 已在本地运行，且已 pull `qwen2.5:7b`

In [6]:
import json
import requests
import time
import os
import tempfile
from pathlib import Path

In [7]:
# ── 配置 ──────────────────────────────────────────────
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL      = "qwen2.5:7b"

# 修改为你的 JSON 文件路径（相对或绝对均可）
JSON_FILES = [
    "public/data/adj_adv.json",
    "public/data/adv_phrasen.json",
    "public/data/nomen_obj.json",
    "public/data/nomen_people.json",
    "public/data/verben_base.json",
    "public/data/verben_phrasen.json",
]

# 跳过例句为空的条目（无内容可翻译）
SKIP_EMPTY_SENTENCE = True
# ──────────────────────────────────────────────────────

In [8]:
def translate(sentence: str) -> str:
    """调用本地 Ollama 翻译一条德语例句为中文，只返回译文本身。"""
    prompt = (
        "请将以下德语句子翻译成中文。"
        "只输出中文译文，不要任何解释、标注或多余文字。\n\n"
        f"{sentence}"
    )
    payload = {
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.1}   # 低温度，翻译更稳定
    }
    resp = requests.post(OLLAMA_URL, json=payload, timeout=60)
    resp.raise_for_status()
    return resp.json()["response"].strip()


def save_json(path: str, data: list):
    """原子写入，避免中途崩溃损坏文件。"""
    dir_name = os.path.dirname(path) or "."
    with tempfile.NamedTemporaryFile("w", encoding="utf-8",
                                     delete=False, dir=dir_name,
                                     suffix=".tmp") as tmp:
        json.dump(data, tmp, ensure_ascii=False, indent=2)
        tmp_path = tmp.name
    os.replace(tmp_path, path)


# 快速检查 Ollama 是否在线
try:
    r = requests.get("http://localhost:11434", timeout=3)
    print("✅ Ollama 在线")
except Exception as e:
    print(f"❌ 无法连接 Ollama：{e}")
    print("请先运行 `ollama serve`")

✅ Ollama 在线


In [9]:
# ── 统计各文件待翻译数量 ──────────────────────────────
for path in JSON_FILES:
    if not Path(path).exists():
        print(f"⚠️  文件不存在: {path}")
        continue
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    need = sum(1 for e in data if e.get("翻译", "") == "")
    print(f"{Path(path).name:30s}  待翻译: {need:4d} / {len(data)}")

adj_adv.json                    待翻译:    3 / 211
adv_phrasen.json                待翻译:    4 / 151
nomen_obj.json                  待翻译:   10 / 833
nomen_people.json               待翻译:    0 / 94
verben_base.json                待翻译:   22 / 324
verben_phrasen.json             待翻译:    6 / 68


In [10]:
# ── 主循环：逐文件、逐条翻译 ─────────────────────────
# 每翻译 SAVE_EVERY 条自动保存一次，防止中途崩溃丢失进度
SAVE_EVERY = 20

for path in JSON_FILES:
    if not Path(path).exists():
        print(f"\n⚠️  跳过（文件不存在）: {path}")
        continue

    data = json.loads(Path(path).read_text(encoding="utf-8"))
    pending = [e for e in data if e.get("翻译", "") == ""]

    if not pending:
        print(f"\n✅ {Path(path).name}：无需翻译，跳过")
        continue

    print(f"\n📄 {Path(path).name}  （{len(pending)} 条待翻译）")
    print("-" * 60)

    done = 0
    errors = 0

    for entry in pending:
        sentence = entry.get("例句", "").strip()

        if SKIP_EMPTY_SENTENCE and not sentence:
            continue  # 例句也为空，无法翻译，跳过

        try:
            translation = translate(sentence)
            entry["翻译"] = translation
            done += 1
            # 显示关键字段（不同 JSON 字段名不同，取第一个非空中文 key）
            word_key = (entry.get("原型") or entry.get("单数") or
                        entry.get("单数男") or entry.get("单词") or "?")
            print(f"  [{done:4d}] {word_key:20s}  →  {translation[:40]}")
        except Exception as e:
            errors += 1
            print(f"  ❌ 翻译失败（{entry.get('原型','?')}）: {e}")
            time.sleep(2)  # 出错后稍等再继续
            continue

        # 定期保存
        if done % SAVE_EVERY == 0:
            save_json(path, data)
            print(f"  💾 已保存进度（{done} 条）")

    # 文件处理完毕，最终保存
    save_json(path, data)
    print(f"  ✅ 完成：{done} 条翻译，{errors} 条失败 → 已保存 {path}")


📄 adj_adv.json  （3 条待翻译）
------------------------------------------------------------
  [   1] ehrenamtlich          →  现在你提到，你也在做志愿工作？
  [   2] einige                →  如一些大的水茎捕手。
  [   3] wunderschön           →  这座城市非常美丽。
  ✅ 完成：3 条翻译，0 条失败 → 已保存 public/data/adj_adv.json

📄 adv_phrasen.json  （4 条待翻译）
------------------------------------------------------------
  [   1] ?                     →  磁带正流行起来。
  [   2] ?                     →  在 school 每三个学生中就有一个患有抑郁。
  [   3] ?                     →  但是有一个条件。
  [   4] ?                     →  -再见，下次见。
  ✅ 完成：4 条翻译，0 条失败 → 已保存 public/data/adv_phrasen.json

📄 nomen_obj.json  （10 条待翻译）
------------------------------------------------------------
  [   1] der Chor              →  合唱：
  [   2] die Kiste             →  有趣的是，观众们后来讲述说有一个玻璃箱，但实际上并不存在箱子。
  [   3] die Matratze          →  就像我准备好躺在垃圾堆上的床垫上睡觉以取得成功一样。
  [   4] das Projekt           →  莉娜说她正在做一个新项目。
  [   5] die Redewendung       →  但这不仅仅是我们迈耶主义者愿意听到的一句口头禅。
  [   6] der Hafen           

In [11]:
# ── 完成后验收：统计剩余空翻译 ───────────────────────
print("\n=== 最终统计 ===")
for path in JSON_FILES:
    if not Path(path).exists():
        continue
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    empty = sum(1 for e in data if e.get("翻译", "") == "")
    status = "✅" if empty == 0 else f"⚠️  还剩 {empty} 条"
    print(f"  {Path(path).name:30s}  {status}")


=== 最终统计 ===
  adj_adv.json                    ✅
  adv_phrasen.json                ✅
  nomen_obj.json                  ✅
  nomen_people.json               ✅
  verben_base.json                ✅
  verben_phrasen.json             ⚠️  还剩 6 条
